# Character N-Gram Language Model (from scratch)

Industrial-grade *lightweight* Generative AI baseline:
- deterministic preprocessing
- train/val split
- Laplace smoothing
- perplexity evaluation
- text generation with temperature

This is designed to run fast on CPU and **save outputs inside the notebook**.

In [1]:
import math
import random
from collections import Counter, defaultdict

SEED = 1337
random.seed(SEED)

# small built-in corpus (public domain-ish style + technical text)
CORPUS = (
    'In the beginning we built systems, and the systems built habits. '
    'A good engineer measures twice, cuts once, and writes tests. '
    'Security is a process, not a product; telemetry is the nervous system. '
    'Machine learning is statistics with compute; MLOps is software engineering with constraints. '
    'When data drifts, models rot. When interfaces drift, teams rot. '
    'The map is not the territory, but it helps you navigate the fog. '
) * 40

def normalize(text: str) -> str:
    text = text.lower()
    # keep only basic ascii chars for reproducibility
    text = ''.join(ch if 32 <= ord(ch) <= 126 else ' ' for ch in text)
    # collapse whitespace
    while '  ' in text:
        text = text.replace('  ', ' ')
    return text.strip()

text = normalize(CORPUS)
len(text), text[:120]

(16759,
 'in the beginning we built systems, and the systems built habits. a good engineer measures twice, cuts once, and writes t')

## Train / validation split

In [2]:
split = int(len(text) * 0.9)
train_text = text[:split]
val_text = text[split:]
len(train_text), len(val_text)

(15083, 1676)

## Build n-gram counts

In [3]:
N = 5  # 5-gram
BOS = ''  # start token
EOS = ''  # end token

def add_markers(s: str) -> str:
    return BOS * (N - 1) + s + EOS

train_seq = add_markers(train_text)

ctx_counts = Counter()
ng_counts = Counter()
vocab = set(train_seq)

for i in range(N - 1, len(train_seq)):
    ctx = train_seq[i - (N - 1): i]
    ch = train_seq[i]
    ctx_counts[ctx] += 1
    ng_counts[(ctx, ch)] += 1

vocab = sorted(vocab)
len(vocab), vocab[:10]

(27, ['\x02', '\x03', ' ', ',', '.', ';', 'a', 'b', 'c', 'd'])

## Smoothed probabilities + perplexity

In [4]:
alpha = 0.5  # Laplace smoothing strength
V = len(vocab)

def prob(ctx: str, ch: str) -> float:
    num = ng_counts[(ctx, ch)] + alpha
    den = ctx_counts[ctx] + alpha * V
    return num / den

def perplexity(s: str) -> float:
    seq = add_markers(s)
    logp = 0.0
    n = 0
    for i in range(N - 1, len(seq)):
        ctx = seq[i - (N - 1): i]
        ch = seq[i]
        p = prob(ctx, ch)
        logp += math.log(p)
        n += 1
    return math.exp(-logp / max(1, n))

ppl_train = perplexity(train_text[:5000])
ppl_val = perplexity(val_text[:2000])
ppl_train, ppl_val

(1.4417278768532868, 1.451186078041222)

## Generation

In [5]:
def sample_next(ctx: str, temperature: float = 1.0) -> str:
    # softmax over log-probs with temperature
    logits = []
    for ch in vocab:
        p = prob(ctx, ch)
        logits.append(math.log(p + 1e-12) / max(1e-6, temperature))
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]
    Z = sum(exps)
    r = random.random()
    c = 0.0
    for ch, e in zip(vocab, exps):
        c += e / Z
        if r <= c:
            return ch
    return vocab[-1]

def generate(max_len: int = 400, temperature: float = 0.9) -> str:
    ctx = BOS * (N - 1)
    out = []
    for _ in range(max_len):
        ch = sample_next(ctx, temperature=temperature)
        if ch == EOS:
            break
        out.append(ch)
        ctx = (ctx + ch)[- (N - 1):]
    return ''.join(out)

for temp in [0.6, 0.9, 1.2]:
    print(f"\n--- temperature {temp} ---")
    print(generate(temperature=temp))


--- temperature 0.6 ---
inal.tesvcy;nv te

--- temperature 0.9 ---
odessntmgwscovgb;

--- temperature 1.2 ---
ini


## Diagnostics: most predictive contexts

In [6]:
# show contexts where the model is most confident
best = []
for ctx, ccount in ctx_counts.most_common(2000):
    # find argmax next char probability
    probs = [(prob(ctx, ch), ch) for ch in vocab]
    pmax, chmax = max(probs, key=lambda x: x[0])
    best.append((pmax, ctx, chmax, ccount))

best = sorted(best, reverse=True)[:12]
[(round(p,3), ctx.replace(BOS,'^'), ch, n) for (p,ctx,ch,n) in best]

[(0.943, ' the', ' ', 216),
 (0.893, 'yste', 'm', 108),
 (0.893, 'syst', 'e', 108),
 (0.893, ' sys', 't', 108),
 (0.848, 'y is', ' ', 72),
 (0.848, 'with', ' ', 72),
 (0.848, 'when', ' ', 72),
 (0.848, 'uilt', ' ', 72),
 (0.848, 'th c', 'o', 72),
 (0.848, 's ro', 't', 72),
 (0.848, 'rot.', ' ', 72),
 (0.848, 'ning', ' ', 72)]